## 0. Setup Ambiente (Locale / Colab)
Rileva automaticamente se stiamo eseguendo su Google Colab e prepara il filesystem ad alte prestazioni per evitare i colli di bottiglia di Google Drive.

In [ ]:
import sys
import os
import shutil

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Ambiente Colab rilevato. Montaggio di Google Drive in corso...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Percorso base del progetto su Drive (modificare se la cartella ha un nome diverso)
    BASE_DIR = '/content/drive/MyDrive/Progetto Machine Learning/Image-Based-Waste-Classification'
    
    # Installa le dipendenze mancanti su Colab
    print("Installazione dipendenze (rich, pyyaml)...")
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'rich', 'pyyaml'])
    
    if os.path.exists(BASE_DIR):
        os.chdir(BASE_DIR)
        sys.path.append(BASE_DIR)
        print(f"Directory di lavoro impostata su: {BASE_DIR}")
    else:
        print(f"\nATTENZIONE: La cartella {BASE_DIR} non esiste su Drive!")
        print("Modifica la variabile BASE_DIR per puntare alla cartella in cui hai salvato il progetto.")
else:
    print("Ambiente locale rilevato. Nessuna operazione speciale richiesta.")


In [ ]:
# Importa librerie, widget e funzioni del modulo locale.

import sys
import os
from pathlib import Path
from datetime import datetime
import json
import yaml

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import models, transforms, datasets
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from tqdm.notebook import tqdm
from PIL import Image

sys.path.append('.')
from waste_classifier.trainer import (
    set_global_seed, worker_init_fn,
    AdaptiveAugmentationDataset, ModelFactory, Trainer, ExperimentManager,
    FocalLoss, build_criterion,
    extract_dataset, analyze_dataset, advanced_stratified_split, analyze_dataset_with_rich, print_dataset_structure_with_rich
)

# Seleziona automaticamente GPU se disponibile, altrimenti CPU.

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
import waste_classifier.calibration as calib


In [ ]:
# Caricamento dei default dal file config.yaml
try:
    with open('config.yaml', 'r') as f:
        default_config = yaml.safe_load(f)
        NUM_WORKERS = default_config.get('training', {}).get('num_workers', 4)
        print(f"[Info] DataLoader configurerà {NUM_WORKERS} workers.")
        INPUT_SIZES = {'efficientnet_b0_ca': 224, 'efficientnet_b0': 224, 'efficientnet_b2': 288, 'efficientnet_b3': 300, 'mobilenet_v3_small': 224}
    print("config.yaml caricato correttamente. I widget useranno questi valori come default.")
except Exception as e:
    print("Impossibile caricare config.yaml, verranno usati i default di base.")
    default_config = {}


## 1 Configurazione Dataset

In [ ]:
# Crea i widget per scegliere dataset, ZIP, percentuali di split e seed.
dataset_path_widget = widgets.Text(
    value='/content/dataset_locale/waste_type_identification' if 'google.colab' in sys.modules else './dataset/waste_type_identification',
    description='Dataset:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

zip_path_widget = widgets.Text(
    value='./waste_type_identification.zip',
    description='ZIP:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

train_ratio_widget = widgets.FloatSlider(
    value=default_config.get('dataset', {}).get('train_ratio', 0.70), min=0.50, max=0.90, step=0.05,
    description='Train %:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

random_seed_widget = widgets.IntText(
    value=default_config.get('dataset', {}).get('random_seed', 42),
    description='Seed:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

extract_button = widgets.Button(
    description='Carica Dataset',
    button_style='primary',
    layout=widgets.Layout(width='auto')
)

# Estrae o carica il dataset, poi calcola classi e numero di immagini.
dataset_output = widgets.Output()

def on_extract_clicked(b):
    with dataset_output:
        clear_output()
        global DATASET_PATH, DATASET_STATS, CLASS_NAMES, N_CLASSES
        
        zip_path = zip_path_widget.value
        dataset_name = 'waste_type_identification'
        if not os.path.exists(zip_path):
            print(f"File {zip_path} non trovato. Provo a scaricarlo automaticamente...")
            import subprocess
            try:
                subprocess.run([sys.executable, "scarica_dataset.py"], check=True)
            except Exception as e:
                print(f"Errore durante il download: {e}")
            if not os.path.exists(zip_path):
                print(f"ERRORE CRITICO: Il file {zip_path} non esiste e il download è fallito.")
                return
        
        
        IN_COLAB = 'google.colab' in sys.modules
        if IN_COLAB:
            extract_to = '/content/dataset_locale'
            print("Utilizzo memoria locale Colab per l'estrazione ultraveloce...")
            local_zip = '/content/temp_dataset.zip'
            if not os.path.exists(local_zip):
                import shutil
                print("Copia dello ZIP da Drive a locale in corso (richiede qualche istante)...")
                shutil.copyfile(zip_path, local_zip)
            zip_path = local_zip
        else:
            extract_to = './dataset'
            
        DATASET_PATH = extract_dataset(zip_path, extract_to, dataset_name)
        DATASET_STATS = analyze_dataset(DATASET_PATH)
        CLASS_NAMES = DATASET_STATS['class_names']
        N_CLASSES = DATASET_STATS['num_classes']
        
        print(f"Dataset caricato con successo!")
        print_dataset_structure_with_rich(str(DATASET_PATH))
        global full_dataset, targets, indices, train_indices, test_indices, val_indices, stratify_labels
        full_dataset = datasets.ImageFolder(DATASET_PATH, transform=None)
        targets = np.array(full_dataset.targets)
        indices = list(range(len(full_dataset)))
        
        train_ratio = train_ratio_widget.value
        train_indices, test_indices, val_indices, stratify_labels = advanced_stratified_split(
            str(DATASET_PATH), full_dataset.samples, split_type='static',
            train_ratio=train_ratio, random_seed=random_seed_widget.value
        )
        
        print(f"Split avanzato completato:")
        analyze_dataset_with_rich(stratify_labels, train_indices, test_indices, val_indices)

extract_button.on_click(on_extract_clicked)

display(widgets.VBox([
    widgets.HTML('<h3>Configurazione Dataset</h3>'),
    zip_path_widget,
    dataset_path_widget,
    widgets.VBox([train_ratio_widget, random_seed_widget]),
    extract_button,
    dataset_output
]))

## 2 Selezione Modello e Configurazione Blocchi

### Come funziona la configurazione dei blocchi

Le reti EfficientNet sono composte da **8 blocchi** (indicizzati da 0 a 7):

```
Input → [Block 0] → [Block 1] → ... → [Block 5] → [Block 6] → [Block 7] → Classifier
         │                                                              │
         └────────────────── Feature Extraction ─────────────────────────┘
```

| Blocchi | Cosa apprendono |
|---------|----------------|
**0-2** | Feature di base (bordi, colori, texture semplici) |
**3-4** | Feature intermedie (forme, pattern) |
**5-7** | Feature avanzate (oggetti, parti specifiche) |

**Regola generale:**
- **Blocchi bassi (0-2)** → quasi sempre congiati (feature universali)
- **Blocchi alti (5-7)** → da sbloccare per adattare al dominio specifico

Seleziona quali blocchi **sbloccare** per il fine-tuning:

In [ ]:
# Crea i controlli per scegliere modello, pesi pre-addestrati e dropout.
model_selector = widgets.SelectMultiple(
    options=['efficientnet_b0', 'efficientnet_b2', 'efficientnet_b3', 'mobilenet_v3_small', 'efficientnet_b0_ca'],
    value=['efficientnet_b0'],
    description='Modelli:',
    layout=widgets.Layout(width='auto', height='120px'),
    style={'description_width': 'initial'}
)

pretrained_checkbox = widgets.Checkbox(
    value=default_config.get('models', {}).get('efficientnet_b0', {}).get('pretrained', True),
    description='Usa pesi pre-addestrati',
    style={'description_width': 'initial'}
)

dropout_widget = widgets.FloatSlider(
    value=default_config.get('models', {}).get('efficientnet_b0', {}).get('dropout', 0.3), min=0.0, max=0.5, step=0.05,
    description='Dropout:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

effnet_options = [
    ('Blocco 1 (Stem)', 0),
    ('Blocco 2', 1),
    ('Blocco 3', 2),
    ('Blocco 4', 3),
    ('Blocco 5', 4),
    ('Blocco 6', 5),
    ('Blocco 7', 6),
    ('Blocco 8', 7),
    ('Blocco 9 (Conv Head)', 8)
]

unfreeze_efficientnet = widgets.SelectMultiple(
    options=effnet_options,
    value=[7, 8],
    description='Sblocca blocchi:',
    layout=widgets.Layout(width='auto', height='150px'),
    style={'description_width': 'initial'}
)

unfreeze_mobilenet = widgets.IntSlider(
    value=3, min=1, max=10, step=1,
    description='Ultimi N blocchi MobileNet:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

model_config_output = widgets.Output()
unfreeze_header = widgets.HTML('<h4>Configurazione Blocchi (Fine-Tuning)</h4>')
unfreeze_instruction = widgets.HTML('<small><i>(Tieni premuto <b>Ctrl</b> o <b>Cmd</b> cliccando per selezionare più blocchi singoli)</i></small>')

def update_model_config(change=None):
# Mostra un riepilogo della configurazione scelta prima del training.
    is_pretrained = pretrained_checkbox.value
    models_selected = list(model_selector.value)
    
    has_efficientnet = any(('efficientnet' in m) for m in models_selected)
    has_mobilenet = any('mobilenet' in m for m in models_selected)
    
    unfreeze_efficientnet.layout.display = 'flex' if (is_pretrained and has_efficientnet) else 'none'
    unfreeze_mobilenet.layout.display = 'flex' if (is_pretrained and has_mobilenet) else 'none'
    unfreeze_instruction.layout.display = 'block' if (is_pretrained and has_efficientnet) else 'none'
    unfreeze_header.layout.display = 'block' if (is_pretrained and (has_efficientnet or has_mobilenet)) else 'none'

    with model_config_output:
        clear_output()
        print(f"Modelli selezionati: {models_selected}")
        print(f"Pretrained: {is_pretrained}")
        print(f"Dropout: {dropout_widget.value}")
        
        for m in models_selected:
            if not is_pretrained:
                print(f"{m}:")
                print("RETE VUOTA: Tutti i blocchi sono sbloccati per un addestramento completo da zero.")
            elif m == 'efficientnet_b0_ca':
                unfrozen_display = [i+1 for i in sorted(unfreeze_efficientnet.value)]
                frozen_display = [i+1 for i in range(9) if (i+1) not in unfrozen_display]
                print(f"{m}:")
                print(f"Blocchi CONGELATI: {frozen_display}")
                print(f"Blocchi SBLOCCATI: {unfrozen_display} + Coordinate Attention")
                if 7 not in unfrozen_display:
                    print("  ⚠️ SUGGERIMENTO: La Coordinate Attention è iniettata tra il Blocco 6 e 7.")
                    print("  Per risultati ottimali, si consiglia di sbloccare almeno il Blocco 7 in su.")
            elif 'efficientnet' in m:
                unfrozen_display = [i+1 for i in sorted(unfreeze_efficientnet.value)]
                frozen_display = [i+1 for i in range(9) if (i+1) not in unfrozen_display]
                print(f"{m}:")
                print(f"Blocchi CONGELATI: {frozen_display}")
                print(f"Blocchi SBLOCCATI: {unfrozen_display}")
            elif 'mobilenet' in m:
                print(f"{m}:")
                print(f"Ultimi {unfreeze_mobilenet.value} blocchi sbloccati")

model_selector.observe(update_model_config)
pretrained_checkbox.observe(update_model_config)
dropout_widget.observe(update_model_config)
unfreeze_efficientnet.observe(update_model_config)
unfreeze_mobilenet.observe(update_model_config)

# Initialize display state
update_model_config()

display(widgets.VBox([
    widgets.HTML('<h3>Selezione Modello</h3>'),
    model_selector,
    pretrained_checkbox,
    dropout_widget,
    unfreeze_header,
    widgets.VBox([unfreeze_instruction, unfreeze_efficientnet, unfreeze_mobilenet]),
    model_config_output
]))



## 3 Configurazione Augmentation

In [ ]:
# Crea i controlli per configurare la data augmentation.
enable_augmentation = widgets.Checkbox(
    value=default_config.get('augmentation', {}).get('enabled', True),
    description='Abilita Augmentation',
    style={'description_width': 'initial'}
)

global_aug_p = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('global_aug_p', 0.6), min=0.0, max=1.0, step=0.05,
    description='Prob. di Augmentation Globale:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

crop_scale_min = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('crop_scale_min', 0.85), min=0.5, max=1.0, step=0.05,
    description='Crop Min Scale:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

crop_scale_max = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('crop_scale_max', 1.0), min=0.5, max=1.0, step=0.05,
    description='Crop Max Scale:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

flip_p = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('flip_p', 0.5), min=0.0, max=1.0, step=0.1,
    description='Prob. Flip Orizz.:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

jitter_p = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('jitter_p', 0.5), min=0.0, max=1.0, step=0.05,
    description='Prob. Jitter Colore:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

brightness = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('brightness', 0.15), min=0.0, max=0.5, step=0.05,
    description='Brightness Jitter:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

contrast = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('contrast', 0.12), min=0.0, max=0.5, step=0.05,
    description='Contrast Jitter:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

rotation_p = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('rotation_p', 0.3), min=0.0, max=1.0, step=0.05,
    description='Prob. Rotazione:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

rotation = widgets.IntSlider(
    value=default_config.get('augmentation', {}).get('rotation', 0), min=0, max=45, step=1,
    description='Rotazione (gradi):',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

widgets.HTML('<h4>Augmentation Avanzate (Disattivate di Default)</h4>'),
noise_p = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('noise_p', 0.0), min=0.0, max=1.0, step=0.05,
    description='Prob. Rumore Gaussiano:', layout=widgets.Layout(width='auto'), style={'description_width': 'initial'}
)
noise_std = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('noise_std', 0.1), min=0.0, max=1.0, step=0.01,
    description='Std. Rumore:', layout=widgets.Layout(width='auto'), style={'description_width': 'initial'}
)
affine_p = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('affine_p', 0.0), min=0.0, max=1.0, step=0.05,
    description='Prob. Affine (Reflect):', layout=widgets.Layout(width='auto'), style={'description_width': 'initial'}
)
affine_degrees = widgets.IntSlider(
    value=default_config.get('augmentation', {}).get('affine_degrees', 0), min=0, max=180, step=5,
    description='Gradi Affine:', layout=widgets.Layout(width='auto'), style={'description_width': 'initial'}
)
affine_translate = widgets.FloatSlider(
    value=default_config.get('augmentation', {}).get('affine_translate', 0.0), min=0.0, max=0.5, step=0.05,
    description='Traslazione Affine:', layout=widgets.Layout(width='auto'), style={'description_width': 'initial'}
)
aug_preview = widgets.Output()

def update_aug_preview(change=None):
    is_disabled = not enable_augmentation.value
    global_aug_p.disabled = is_disabled
    jitter_p.disabled = is_disabled
    rotation_p.disabled = is_disabled
    noise_p.disabled = is_disabled
    noise_std.disabled = is_disabled
    affine_p.disabled = is_disabled
    affine_degrees.disabled = is_disabled
    affine_translate.disabled = is_disabled
    crop_scale_min.disabled = is_disabled
    crop_scale_max.disabled = is_disabled
    flip_p.disabled = is_disabled
    brightness.disabled = is_disabled
    contrast.disabled = is_disabled
    rotation.disabled = is_disabled
    
    with aug_preview:
        clear_output()
        if is_disabled:
            print('Augmentation DISATTIVATA. Verrà usato solo il resize+crop canonico.')
        else:
            print('Strategie di augmentation ATTIVE (uniformi su tutte le classi):')
            print(f' - RandomResizedCrop scale: ({crop_scale_min.value:.2f}, {crop_scale_max.value:.2f})')
            print(f' - RandomHorizontalFlip prob: {flip_p.value:.2f}')
            print(f' - ColorJitter: brightness={brightness.value:.2f}, contrast={contrast.value:.2f}')
            if rotation.value > 0:
                print(f' - RandomRotation: ±{rotation.value}° (con fill bianco)')
            else:
                print(' - Nessuna Rotazione applicata.')

enable_augmentation.observe(update_aug_preview)
global_aug_p.observe(update_aug_preview)
jitter_p.observe(update_aug_preview)
rotation_p.observe(update_aug_preview)
noise_p.observe(update_aug_preview)
noise_std.observe(update_aug_preview)
affine_p.observe(update_aug_preview)
affine_degrees.observe(update_aug_preview)
affine_translate.observe(update_aug_preview)
crop_scale_min.observe(update_aug_preview)
crop_scale_max.observe(update_aug_preview)
flip_p.observe(update_aug_preview)
brightness.observe(update_aug_preview)
contrast.observe(update_aug_preview)
rotation.observe(update_aug_preview)

update_aug_preview()

display(widgets.VBox([
    widgets.HTML('<h3>Configurazione Augmentation</h3>'),
    enable_augmentation,
    global_aug_p,
    crop_scale_min,
    crop_scale_max,
    flip_p,
    jitter_p,
    brightness,
    contrast,
    rotation_p,
    rotation,
    widgets.HTML('<h4>Augmentation Avanzate</h4>'),
    noise_p,
    noise_std,
    affine_p,
    affine_degrees,
    affine_translate,
    aug_preview
]))


## 4 Configurazione Training

In [ ]:
# Crea i controlli per batch size, epoche, learning rate e scheduler.
batch_size_widget = widgets.Dropdown(
    options=[16, 32, 64, 128],
    value=default_config.get('training', {}).get('batch_size', 32),
    description='Batch size:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

num_workers_widget = widgets.IntSlider(
        value=default_config.get('training', {}).get('num_workers', 4), min=0, max=16, step=1,
        description='Num Workers:',
        layout=widgets.Layout(width='auto'),
        style={'description_width': 'initial'}
)

use_amp = widgets.Checkbox(
    value=default_config.get('training', {}).get('use_amp', True),
    description='Mixed Precision (AMP)',
    style={'description_width': 'initial'}
)

fe_epochs = widgets.IntSlider(
    value=default_config.get('training', {}).get('feature_extraction', {}).get('epochs', 10), min=1, max=50, step=1,
    description='Epoche FE:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

ft_epochs = widgets.IntSlider(
    value=default_config.get('training', {}).get('fine_tuning', {}).get('epochs', 30), min=1, max=100, step=1,
    description='Epoche FT:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

lr_fe_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('feature_extraction', {}).get('learning_rate', 1e-3), base=10, min=-6, max=-1, step=0.1,
    description='LR Feature Extraction:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

lr_ft_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('fine_tuning', {}).get('learning_rate', 1e-5), base=10, min=-7, max=-2, step=0.1,
    description='LR Fine Tuning:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

patience_fe_widget = widgets.IntSlider(
    value=default_config.get('training', {}).get('feature_extraction', {}).get('patience', 5), min=1, max=20, step=1,
    description='Patience FE:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

min_delta_fe_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('feature_extraction', {}).get('min_delta', 1e-3), base=10, min=-5, max=-1, step=0.1,
    description='Min Delta FE:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

patience_ft_widget = widgets.IntSlider(
    value=default_config.get('training', {}).get('fine_tuning', {}).get('patience', 7), min=1, max=20, step=1,
    description='Patience FT:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

min_delta_ft_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('fine_tuning', {}).get('min_delta', 5e-4), base=10, min=-5, max=-1, step=0.1,
    description='Min Delta FT:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

weight_decay_widget = widgets.FloatLogSlider(
    value=default_config.get('training', {}).get('weight_decay', 1e-4), base=10, min=-6, max=-1, step=0.1,
    description='Weight Decay:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

use_scheduler = widgets.Checkbox(
    value=default_config.get('training', {}).get('scheduler', {}).get('enabled', True),
# Configura il tipo di scheduler da usare durante il training.
    description='Usa Scheduler LR',
    style={'description_width': 'initial'}
)

scheduler_type = widgets.Dropdown(
    options=['ReduceLROnPlateau', 'CosineAnnealingLR', 'StepLR'],
    value=default_config.get('training', {}).get('scheduler', {}).get('type', 'ReduceLROnPlateau'),
description='Tipo scheduler:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

scheduler_factor = widgets.FloatSlider(
    value=default_config.get('training', {}).get('scheduler', {}).get('factor', 0.3), min=0.1, max=0.9, step=0.1,
    description='Factor riduzione:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

scheduler_patience = widgets.IntSlider(
        value=default_config.get('training', {}).get('scheduler', {}).get('patience', 2), min=1, max=10, step=1,
        description='Patience Scheduler:',
        layout=widgets.Layout(width='auto'),
        style={'description_width': 'initial'}
)

use_kfold = widgets.Checkbox(
    value=default_config.get('kfold', {}).get('enabled', True),
    description='Usa K-Fold',
# Configura la validazione K-Fold, se abilitata.
    style={'description_width': 'initial'}
)

kfold_mode = widgets.Dropdown(
    options=['Valutativo + Modello Singolo', 'Ensemble di Modelli'],
    value='Valutativo + Modello Singolo',
    description='Modalità KF:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

n_splits = widgets.IntSlider(
    value=default_config.get('kfold', {}).get('n_splits', 3), min=2, max=10, step=1,
    description='N folds:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

use_focal_loss = widgets.Checkbox(
    value=default_config.get('training', {}).get('focal_loss', {}).get('enabled', False),
    description='Usa Focal Loss',
    style={'description_width': 'initial'}
)

focal_gamma_widget = widgets.FloatSlider(
    value=default_config.get('training', {}).get('focal_loss', {}).get('gamma', 2.0), min=0.0, max=5.0, step=0.5,
    description='Focal gamma (γ):',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

def update_training_config(change=None):
    show_scheduler = use_scheduler.value
    scheduler_type.layout.display = 'flex' if show_scheduler else 'none'
    scheduler_factor.layout.display = 'flex' if show_scheduler else 'none'
    scheduler_patience.layout.display = 'flex' if show_scheduler else 'none'
    
    show_kfold = use_kfold.value
    n_splits.layout.display = 'flex' if show_kfold else 'none'
    kfold_mode.layout.display = 'flex' if show_kfold else 'none'
    focal_gamma_widget.layout.display = 'flex' if use_focal_loss.value else 'none'

use_scheduler.observe(update_training_config)
use_kfold.observe(update_training_config)
use_focal_loss.observe(update_training_config)

# Inizializza lo stato visivo
update_training_config()


use_weighted_sampler = widgets.Checkbox(
    value=False,
    description='Usa Weighted Random Sampler (oversampling classi minoritarie)',
    tooltip='Se attivato, bilancia le classi pescando piu spesso quelle minoritarie.',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

display(widgets.VBox([
    widgets.HTML('<h3>Configurazione Training</h3>'),
    widgets.HTML('<h4>Base</h4>'),
    widgets.VBox([batch_size_widget, num_workers_widget, use_amp]),
    use_weighted_sampler,
    widgets.VBox([fe_epochs, ft_epochs]),
    lr_fe_widget,
    lr_ft_widget,
    patience_fe_widget,
    min_delta_fe_widget,
    patience_ft_widget,
    min_delta_ft_widget,
    weight_decay_widget,
    widgets.HTML('<h4>Funzione di Loss</h4>'),
    use_focal_loss,
    focal_gamma_widget,
    widgets.HTML('<h4>Scheduler</h4>'),
    use_scheduler,
    scheduler_type,
    scheduler_factor,
    scheduler_patience,
    widgets.HTML('<h4>K-Fold Cross Validation</h4>'),
    use_kfold,
    n_splits,
    kfold_mode
]))


## 5 Nome Esperimento

In [ ]:
# Crea i controlli per nome esperimento e cartella di output.
experiment_name = widgets.Text(
    value='',
    description='Nome:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'},
placeholder='Opzionale - es: test_b0_augmented'
)

output_base_dir = widgets.Text(
    value='./experiments',
    description='Cartella output:',
    layout=widgets.Layout(width='auto'),
    style={'description_width': 'initial'}
)

estim_time = widgets.Output()

def estimate_time(change=None):
    with estim_time:
        clear_output()
        models_count = len(model_selector.value)
        total_epochs = fe_epochs.value + ft_epochs.value
# Aggiorna una stima indicativa del tempo in base ai parametri correnti.
        if use_kfold.value:
            total_epochs *= n_splits.value
        if use_scheduler.value:
            total_epochs = int(total_epochs * 0.7)  # Stima con early stopping
        
        sec_per_epoch = 30
        total_sec = total_epochs * sec_per_epoch * models_count
        total_min = total_sec / 60
        
        print(f"⏱ Tempo stimato: ~{total_min:.0f} minuti ({total_min/60:.1f} ore)")
        print(f"Modelli: {models_count}")
        print(f"Epoche totali (con FE+FT{'+KFOLD' if use_kfold.value else ''}): {total_epochs}")

model_selector.observe(estimate_time)
fe_epochs.observe(estimate_time)
ft_epochs.observe(estimate_time)
use_kfold.observe(estimate_time)
n_splits.observe(estimate_time)
use_scheduler.observe(estimate_time)

display(widgets.VBox([
    widgets.HTML('<h3>Esperimento</h3>'),
    experiment_name,
    output_base_dir,
    estim_time
]))

## 6 Avvio Training

In [ ]:
def run_training(b=None):
    from waste_classifier.trainer import set_global_seed
    set_global_seed(random_seed_widget.value)
    print(f"\n[Setup] Global Seed impostato a: {random_seed_widget.value} per la massima riproducibilità.")
    import csv
    from pathlib import Path
    def save_dataset_protocol_csv(exp_path, ds_samples, c_names, tr_idx, v_idx, te_idx, filename="dataset_splits.csv"):
        csv_path = exp_path / filename
        with open(csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["filepath", "split", "class_name", "class_id"])
            split_map = {}
            for idx in tr_idx: split_map[idx] = "train"
            for idx in v_idx: split_map[idx] = "val"
            for idx in te_idx: split_map[idx] = "test"
            for i, (path, c_id) in enumerate(ds_samples):
                split = split_map.get(i, "unknown")
                rel_path = "/".join(Path(path).parts[-2:])
                writer.writerow([rel_path, split, c_names[c_id], c_id])
        print(f"\n[Info] Protocollo Split salvato in: {csv_path}")

    import time
    total_start_time = time.time()
    undiff_idx = -1
    carveout_indices = []
    for i_cls, name in enumerate(CLASS_NAMES):
        name_lower = name.lower()
        if 'undifferentiated' in name_lower or 'other' in name_lower:
            undiff_idx = i_cls
        if 'plastic' in name_lower or 'glass' in name_lower or 'metal' in name_lower:
            carveout_indices.append(i_cls)
    if undiff_idx == -1:
        raise ValueError("Attenzione: impossibile trovare la classe 'undifferentiated' o 'other'. Il routing OSR produrrà predizioni a -1 facendo fallire la validazione. Controlla i nomi delle classi nel dataset.")
    print(f'[Info OSR] Undifferentiated Index: {undiff_idx}')
    print(f'[Info OSR] Carve-out Indices (Plastic/Glass/Metal): {carveout_indices}')
    fold_taus = []
    fold_temps = []
    global trainer
    
    if 'DATASET_PATH' not in globals():
        print("Devi prima caricare il dataset!")
        return
    
    models_selected = list(model_selector.value)
    if not models_selected:
        print("Seleziona almeno un modello!")
        return
    
    print("=" * 60)
    print("AVVIO TRAINING")
    print("=" * 60)
    
    exp_manager = ExperimentManager(output_base_dir.value)
    
    for model_name in models_selected:
        print(f"\n{'='*60}")
        print(f"Modello: {model_name}")
        print(f"{'='*60}")
        
        exp_dir = exp_manager.create_experiment_dir(
            model_name, 
            experiment_name.value if experiment_name.value else None
        )
        print(f"Esperimento: {exp_dir}")
        
        config = {
            'model': {
                'name': model_name,
                'pretrained': pretrained_checkbox.value,
                'dropout': dropout_widget.value,
            },
            'training': {
                'batch_size': batch_size_widget.value,
                'use_amp': use_amp.value,
                'fe_epochs': fe_epochs.value,
                'ft_epochs': ft_epochs.value,
                'lr_fe': lr_fe_widget.value,
                'lr_ft': lr_ft_widget.value,
                'patience_fe': patience_fe_widget.value,
                'min_delta_fe': min_delta_fe_widget.value,
                'patience_ft': patience_ft_widget.value,
                'min_delta_ft': min_delta_ft_widget.value,
                'weight_decay': weight_decay_widget.value,
                'label_smoothing': default_config.get('training', {}).get('label_smoothing', 0.0),
                'weighted_sampler': use_weighted_sampler.value,
                'focal_loss': use_focal_loss.value,
                'focal_gamma': focal_gamma_widget.value,
            },
            'augmentation': {
                'enabled': enable_augmentation.value,
                'global_aug_p': global_aug_p.value if enable_augmentation.value else 0.0,
                'crop_scale_min': crop_scale_min.value,
                'crop_scale_max': crop_scale_max.value,
                'flip_p': flip_p.value,
                'jitter_p': jitter_p.value,
                'rotation_p': rotation_p.value,
                'noise_p': noise_p.value,
                'noise_std': noise_std.value,
                'affine_p': affine_p.value,
                'affine_degrees': affine_degrees.value,
                'affine_translate': affine_translate.value,
                'brightness': brightness.value,
                'contrast': contrast.value,
                'rotation': rotation.value,
            },
            'scheduler': {
                'enabled': use_scheduler.value,
                'type': scheduler_type.value,
                'factor': scheduler_factor.value,
            },
            'kfold': {
                'enabled': use_kfold.value,
                'n_splits': n_splits.value,
            },
            'dataset': {
                'path': str(DATASET_PATH),
                'train_ratio': train_ratio_widget.value,
                'random_seed': random_seed_widget.value,
            }
        }
        if use_focal_loss.value:
            config['training']['label_smoothing'] = 0.0
            print('[Config] Label Smoothing forzato a 0.0 perché Focal Loss è attiva.')
        exp_manager.save_config(config, exp_dir)
        if use_kfold.value and kfold_mode.value == 'Ensemble di Modelli':
            from sklearn.model_selection import StratifiedKFold
            skf_csv = StratifiedKFold(n_splits=n_splits.value, shuffle=True, random_state=random_seed_widget.value)
            
            csv_path = exp_dir / "dataset_splits_ensemble.csv"
            with open(csv_path, "w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                headers = ["filepath", "class_name", "class_id", "test_set"] + [f"fold_{i+1}" for i in range(n_splits.value)]
                writer.writerow(headers)
                
                folds_tr = []
                folds_va = []
                for tr_pos, te_pos in skf_csv.split(trainval_indices, targets[trainval_indices]):
                    folds_tr.append(set(trainval_indices[tr_pos]))
                    folds_va.append(set(trainval_indices[te_pos]))
                    
                test_set_ids = set(test_indices)
                for i, (path, c_id) in enumerate(full_dataset.samples):
                    from pathlib import Path
                    rel_path = "/".join(Path(path).parts[-2:])
                    is_test = "True" if i in test_set_ids else "False"
                    row = [rel_path, CLASS_NAMES[c_id], c_id, is_test]
                    for fold_idx in range(n_splits.value):
                        if i in folds_tr[fold_idx]: row.append("train")
                        elif i in folds_va[fold_idx]: row.append("val")
                        else: row.append("-")
                    writer.writerow(row)
            print(f"\n[Info] Protocollo Split Ensemble salvato in: {csv_path}")
        else:
            save_dataset_protocol_csv(exp_dir, full_dataset.samples, CLASS_NAMES, train_indices, val_indices, test_indices)
        
        # input_size standard per architettura (valori fissi delle specifiche ufficiali)
        input_size = INPUT_SIZES.get(model_name, 224)
        

        trainer = Trainer(config, exp_dir)
        
        if use_kfold.value:
            fold_scores = []
            trainval_indices = np.array(train_indices + val_indices)
            trainval_targets = targets[trainval_indices]
            
            skf = StratifiedKFold(
                n_splits=n_splits.value, 
                shuffle=True, 
                random_state=random_seed_widget.value
            )
            
            for fold, (tr_pos, te_pos) in enumerate(skf.split(trainval_indices, trainval_targets), 1):
                print(f"\n--- Fold {fold}/{n_splits.value} ---")
                
                fold_train_idx = trainval_indices[tr_pos]
                fold_test_idx = trainval_indices[te_pos]
                
                fold_train_ds = AdaptiveAugmentationDataset(
                    Subset(full_dataset, fold_train_idx), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                    resize=input_size + 32, crop=input_size, is_train=True,
                    aug_enabled=enable_augmentation.value,
                    global_aug_p=global_aug_p.value if enable_augmentation.value else 0.0,
                    crop_scale=(crop_scale_min.value, crop_scale_max.value),
                    flip_p=flip_p.value,
                    jitter_p=jitter_p.value,
                    brightness=brightness.value,
                    contrast=contrast.value,
                    rotation_degrees=rotation.value,
                    rotation_p=rotation_p.value,
                    noise_p=noise_p.value,
                    noise_std=noise_std.value,
                    affine_p=affine_p.value,
                    affine_degrees=affine_degrees.value,
                    affine_translate=affine_translate.value
                )
                fold_test_ds = AdaptiveAugmentationDataset(
                    Subset(full_dataset, fold_test_idx), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                    resize=input_size + 32, crop=input_size, is_train=False
                )
                
                if use_weighted_sampler.value:
                    from waste_classifier.trainer import get_weighted_sampler
                    sampler = get_weighted_sampler(fold_train_ds)
                    print(f"[Fold {fold}] Weighted Sampler attivato.")
                    fold_train_loader = DataLoader(
                        fold_train_ds, batch_size=batch_size_widget.value, shuffle=False, sampler=sampler,
                        num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn
                    )
                else:
                    fold_train_loader = DataLoader(
                        fold_train_ds, batch_size=batch_size_widget.value,
                        shuffle=True, num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn
                    )
                fold_test_loader = DataLoader(
                    fold_test_ds, batch_size=batch_size_widget.value,
                    shuffle=False, num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn
                )
                
                model = ModelFactory.create_model(
                    model_name, N_CLASSES, device,
                    pretrained=pretrained_checkbox.value,
                    dropout=dropout_widget.value
                )
                
                fold_weights = trainer.compute_class_weights(fold_train_idx, targets, N_CLASSES)
                fold_criterion = build_criterion(
                    use_focal=use_focal_loss.value,
                    class_weights=None if use_weighted_sampler.value else fold_weights,
                    gamma=focal_gamma_widget.value,
                    label_smoothing=config['training'].get('label_smoothing', 0.0)
                )
                
                fe_params = [p for p in model.parameters() if p.requires_grad]
                opt_fe = optim.AdamW(fe_params, lr=lr_fe_widget.value, weight_decay=weight_decay_widget.value)
                
                hist_fe = trainer.train_model(
                    model, opt_fe, fold_train_loader, fold_test_loader, fold_criterion,
                    epochs=fe_epochs.value, patience=patience_fe_widget.value, min_delta=min_delta_fe_widget.value,
                    save_path=str(exp_dir / f"models/fold{fold}_fe.pth"),
                    use_amp=use_amp.value, phase_name=f"fold{fold}_feature_extraction"
                )
                
                if 'efficientnet' in model_name:
                    ModelFactory.unfreeze_for_finetuning(
                        model, model_name, blocks=unfreeze_efficientnet.value
                    )
                elif 'mobilenet' in model_name:
                    ModelFactory.unfreeze_for_finetuning(
                        model, model_name, n_last_blocks=unfreeze_mobilenet.value
                    )
                
                opt_ft = optim.AdamW(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    lr=lr_ft_widget.value, weight_decay=weight_decay_widget.value
                )
                
                scheduler = None
                if use_scheduler.value:
                    if scheduler_type.value == 'ReduceLROnPlateau':
                        scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt_ft, mode='max', factor=scheduler_factor.value, patience=scheduler_patience.value)
                    elif scheduler_type.value == 'CosineAnnealingLR':
                        scheduler = optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=ft_epochs.value)
                    elif scheduler_type.value == 'StepLR':
                        scheduler = optim.lr_scheduler.StepLR(opt_ft, step_size=10, gamma=scheduler_factor.value)
                
                hist_ft = trainer.train_model(
                    model, opt_ft, fold_train_loader, fold_test_loader, fold_criterion,
                    epochs=ft_epochs.value, patience=patience_ft_widget.value, min_delta=min_delta_ft_widget.value,
                    save_path=str(exp_dir / f"models/fold{fold}_ft.pth"),
                    initial_best=max(hist_fe['val_bal_acc']) if hist_fe.get('val_bal_acc') else 0.0, scheduler=scheduler,
                    use_amp=use_amp.value, phase_name=f"fold{fold}_fine_tuning"
                )
                
                best_fold = max(max(hist_fe['val_bal_acc'] if hist_fe.get('val_bal_acc') else [0.0]), max(hist_ft['val_bal_acc'] if hist_ft.get('val_bal_acc') else [0.0]))
                fold_scores.append(best_fold)
                print(f"Fold {fold} best: {best_fold*100:.2f}%")
                
                exp_manager.save_history(hist_fe, exp_dir, filename=f"fold{fold}_fe_history.json")
                exp_manager.save_history(hist_ft, exp_dir, filename=f"fold{fold}_ft_history.json")
                exp_manager.save_training_curves(hist_ft, exp_dir, f"{model_name}_fold{fold}")
                print(f"\n--- Calibrazione OSR Fold {fold} ---")
                t_fold = calib.fit_temperature(model, fold_test_loader, device)
                tau_fold, _ = calib.find_optimal_threshold(model, fold_test_loader, device, t_fold, undiff_idx, carveout_indices)
                fold_taus.append(tau_fold)
                fold_temps.append(t_fold)
            
            print(f"\n{'='*40}")
            print(f"K-Fold Val Estimation: {np.mean(fold_scores)*100:.2f}% (+/-{np.std(fold_scores)*100:.2f}%)")
            print(f"{'='*40}")
            
            if kfold_mode.value == 'Valutativo + Modello Singolo':
                print("\n=== ADDESTRAMENTO MODELLO FINALE SULL'INTERO TRAINING SET ===")
                from sklearn.model_selection import train_test_split
                final_tr_idx, final_val_idx = train_test_split(
                    trainval_indices, test_size=0.15,
                    stratify=targets[trainval_indices], random_state=random_seed_widget.value
                )
                save_dataset_protocol_csv(exp_dir, full_dataset.samples, CLASS_NAMES, final_tr_idx, final_val_idx, test_indices, filename="dataset_splits_final.csv")
                
                final_train_ds = AdaptiveAugmentationDataset(
                    Subset(full_dataset, final_tr_idx), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                    resize=input_size+32, crop=input_size, is_train=True,
                    aug_enabled=enable_augmentation.value,
                    global_aug_p=global_aug_p.value if enable_augmentation.value else 0.0,
                    crop_scale=(crop_scale_min.value, crop_scale_max.value),
                    flip_p=flip_p.value,
                    jitter_p=jitter_p.value,
                    brightness=brightness.value,
                    contrast=contrast.value,
                    rotation_degrees=rotation.value,
                    rotation_p=rotation_p.value,
                    noise_p=noise_p.value,
                    noise_std=noise_std.value,
                    affine_p=affine_p.value,
                    affine_degrees=affine_degrees.value,
                    affine_translate=affine_translate.value
                )
                final_val_ds = AdaptiveAugmentationDataset(
                    Subset(full_dataset, final_val_idx), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                    resize=input_size+32, crop=input_size, is_train=False
                )
                
                if use_weighted_sampler.value:
                    from waste_classifier.trainer import get_weighted_sampler
                    sampler = get_weighted_sampler(final_train_ds)
                    print("[Fine-Tuning] Weighted Sampler attivato per il training finale.")
                    final_train_loader = DataLoader(
                        final_train_ds, batch_size=batch_size_widget.value, shuffle=False, sampler=sampler,
                        num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn
                    )
                else:
                    final_train_loader = DataLoader(
                        final_train_ds, batch_size=batch_size_widget.value, shuffle=True,
                        num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn
                    )
                final_val_loader = DataLoader(
                    final_val_ds, batch_size=batch_size_widget.value, shuffle=False,
                    num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn
                )
                
                final_weights = trainer.compute_class_weights(final_tr_idx, targets, N_CLASSES)
                final_criterion = build_criterion(
                    use_focal=use_focal_loss.value,
                    class_weights=None if use_weighted_sampler.value else final_weights,
                    gamma=focal_gamma_widget.value,
                    label_smoothing=config['training'].get('label_smoothing', 0.0)
                )
                
                final_model = ModelFactory.create_model(model_name, N_CLASSES, device, pretrained=pretrained_checkbox.value, dropout=dropout_widget.value)
                
                fe_params = [p for p in final_model.parameters() if p.requires_grad]
                opt_fe = optim.AdamW(fe_params, lr=lr_fe_widget.value, weight_decay=weight_decay_widget.value)
                
                hist_fe = trainer.train_model(
                    final_model, opt_fe, final_train_loader, final_val_loader, final_criterion,
                    epochs=fe_epochs.value, patience=patience_fe_widget.value, min_delta=min_delta_fe_widget.value,
                    save_path=str(exp_dir / "models/final_fe.pth"),
                    use_amp=use_amp.value, phase_name="final_feature_extraction"
                )
                
                if 'efficientnet' in model_name:
                    ModelFactory.unfreeze_for_finetuning(final_model, model_name, blocks=unfreeze_efficientnet.value)
                elif 'mobilenet' in model_name:
                    ModelFactory.unfreeze_for_finetuning(final_model, model_name, n_last_blocks=unfreeze_mobilenet.value)
                
                opt_ft = optim.AdamW(
                    filter(lambda p: p.requires_grad, final_model.parameters()),
                    lr=lr_ft_widget.value, weight_decay=weight_decay_widget.value
                )
                
                scheduler = None
                if use_scheduler.value:
                    if scheduler_type.value == 'ReduceLROnPlateau':
                        scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt_ft, mode='max', factor=scheduler_factor.value, patience=scheduler_patience.value)
                    elif scheduler_type.value == 'CosineAnnealingLR':
                        scheduler = optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=ft_epochs.value)
                    elif scheduler_type.value == 'StepLR':
                        scheduler = optim.lr_scheduler.StepLR(opt_ft, step_size=10, gamma=scheduler_factor.value)
                        
                hist_ft = trainer.train_model(
                    final_model, opt_ft, final_train_loader, final_val_loader, final_criterion,
                    epochs=ft_epochs.value, patience=patience_ft_widget.value, min_delta=min_delta_ft_widget.value,
                    save_path=str(exp_dir / "models/final_ft.pth"),
                    initial_best=max(hist_fe['val_bal_acc']) if hist_fe.get('val_bal_acc') else 0.0, scheduler=scheduler,
                    use_amp=use_amp.value, phase_name="final_fine_tuning"
                )
                
                exp_manager.save_history(hist_fe, exp_dir, filename="final_fe_history.json")
                exp_manager.save_history(hist_ft, exp_dir, filename="final_ft_history.json")
                exp_manager.save_training_curves(hist_ft, exp_dir, f"{model_name}_final")
                print(f"\n--- Calibrazione OSR Modello Finale ---")
                best_t = calib.fit_temperature(final_model, final_val_loader, device)
                best_tau, _ = calib.find_optimal_threshold(final_model, final_val_loader, device, best_t, undiff_idx, carveout_indices)
                model = final_model
            else:
                print("\n=== MODALITA' ENSEMBLE SELEZIONATA: SALTO L'ADDESTRAMENTO DEL MODELLO SINGOLO ===")

        else:
            # Usa val_indices come validation durante il training: il test set
            # rimane completamente nascosto fino alla valutazione finale.
            train_ds = AdaptiveAugmentationDataset(
                    Subset(full_dataset, train_indices), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                    resize=input_size+32, crop=input_size, is_train=True,
                    aug_enabled=enable_augmentation.value,
                    global_aug_p=global_aug_p.value if enable_augmentation.value else 0.0,
                    crop_scale=(crop_scale_min.value, crop_scale_max.value),
                    flip_p=flip_p.value,
                    jitter_p=jitter_p.value,
                    brightness=brightness.value,
                    contrast=contrast.value,
                    rotation_degrees=rotation.value,
                    rotation_p=rotation_p.value,
                    noise_p=noise_p.value,
                    noise_std=noise_std.value,
                    affine_p=affine_p.value,
                    affine_degrees=affine_degrees.value,
                    affine_translate=affine_translate.value
                )
            val_ds = AdaptiveAugmentationDataset(
                    Subset(full_dataset, val_indices), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                    resize=input_size+32, crop=input_size, is_train=False
                )
            
            if use_weighted_sampler.value:
                from waste_classifier.trainer import get_weighted_sampler
                sampler = get_weighted_sampler(train_ds)
                print("[Standard Training] Weighted Sampler attivato.")
                train_loader = DataLoader(train_ds, batch_size=batch_size_widget.value, shuffle=False, sampler=sampler, num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn)
            else:
                train_loader = DataLoader(train_ds, batch_size=batch_size_widget.value, shuffle=True, num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn)
            val_loader = DataLoader(val_ds, batch_size=batch_size_widget.value, shuffle=False, num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn)
            
            model = ModelFactory.create_model(model_name, N_CLASSES, device, pretrained=pretrained_checkbox.value, dropout=dropout_widget.value)
            
            train_weights = trainer.compute_class_weights(train_indices, targets, N_CLASSES)
            criterion = build_criterion(
                use_focal=use_focal_loss.value,
                class_weights=None if use_weighted_sampler.value else train_weights,
                gamma=focal_gamma_widget.value,
                label_smoothing=config['training'].get('label_smoothing', 0.0)
            )
            
            fe_params = [p for p in model.parameters() if p.requires_grad]
            opt_fe = optim.AdamW(fe_params, lr=lr_fe_widget.value, weight_decay=weight_decay_widget.value)
            
            hist_fe = trainer.train_model(
                model, opt_fe, train_loader, val_loader, criterion,
                epochs=fe_epochs.value, patience=patience_fe_widget.value, min_delta=min_delta_fe_widget.value,
                save_path=str(exp_dir / "models/final_fe.pth"),
                use_amp=use_amp.value, phase_name="feature_extraction"
            )
            
            if 'efficientnet' in model_name:
                ModelFactory.unfreeze_for_finetuning(model, model_name, blocks=unfreeze_efficientnet.value)
            elif 'mobilenet' in model_name:
                ModelFactory.unfreeze_for_finetuning(model, model_name, n_last_blocks=unfreeze_mobilenet.value)
            
            opt_ft = optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=lr_ft_widget.value, weight_decay=weight_decay_widget.value
            )
            
            scheduler = None
            if use_scheduler.value:
                if scheduler_type.value == 'ReduceLROnPlateau':
                    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt_ft, mode='max', factor=scheduler_factor.value, patience=scheduler_patience.value)
                elif scheduler_type.value == 'CosineAnnealingLR':
                    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=ft_epochs.value)
                elif scheduler_type.value == 'StepLR':
                    scheduler = optim.lr_scheduler.StepLR(opt_ft, step_size=10, gamma=scheduler_factor.value)
                    
            hist_ft = trainer.train_model(
                model, opt_ft, train_loader, val_loader, criterion,
                epochs=ft_epochs.value, patience=patience_ft_widget.value, min_delta=min_delta_ft_widget.value,
                save_path=str(exp_dir / "models/final_ft.pth"),
                initial_best=max(hist_fe['val_bal_acc']) if hist_fe.get('val_bal_acc') else 0.0, scheduler=scheduler,
                use_amp=use_amp.value, phase_name="fine_tuning"
            )
            
            exp_manager.save_history(hist_fe, exp_dir, filename="fe_history.json")
            exp_manager.save_history(hist_ft, exp_dir, filename="ft_history.json")
            exp_manager.save_training_curves(hist_ft, exp_dir, model_name)
            print(f"\n--- Calibrazione OSR Standard ---")
            best_t = calib.fit_temperature(model, val_loader, device)
            best_tau, _ = calib.find_optimal_threshold(model, val_loader, device, best_t, undiff_idx, carveout_indices)
        
        # --- FASE FINALE CONDIVISA: VALUTAZIONE SUL VERO TEST SET ---
        print("\n=== VALUTAZIONE SUL TEST SET FINALE ===")
        final_test_ds = AdaptiveAugmentationDataset(
                    Subset(full_dataset, test_indices), {i: n.lower() for i, n in enumerate(CLASS_NAMES)},
                    resize=input_size+32, crop=input_size, is_train=False
                )
        final_test_loader = DataLoader(
            final_test_ds, batch_size=batch_size_widget.value, shuffle=False,
            num_workers=num_workers_widget.value, pin_memory=True, persistent_workers=(num_workers_widget.value > 0), worker_init_fn=worker_init_fn
        )
        
        y_true, y_pred_raw, y_pred = [], [], []
        if use_kfold.value and kfold_mode.value == 'Ensemble di Modelli':
            print(f"Esecuzione inferenza Ensemble con {n_splits.value} modelli...")
            best_t = 1.0
            best_tau = float(np.mean(fold_taus)) if len(fold_taus) > 0 else 0.5
            ensemble_models = []
            for fold in range(1, n_splits.value + 1):
                m = ModelFactory.create_model(model_name, N_CLASSES, device, pretrained=False)
                m.load_state_dict(torch.load(str(exp_dir / f"models/fold{fold}_ft.pth"), map_location=device))
                m.eval()
                ensemble_models.append(m)
            with torch.no_grad():
                for imgs, labels in final_test_loader:
                    imgs = imgs.to(device)
                    probs = torch.zeros(imgs.size(0), N_CLASSES).to(device)
                    for i, m in enumerate(ensemble_models):
                        probs += torch.softmax(m(imgs) / fold_temps[i], dim=1)
                    probs /= len(ensemble_models)
                    y_true.extend(labels.numpy())
                    y_pred_raw.extend(probs.argmax(dim=1).cpu().numpy())
                    y_pred.extend(calib.apply_reject_routing(torch.log(probs + 1e-8), 1.0, float(np.mean(fold_taus)) if len(fold_taus) > 0 else 0.5, undiff_idx, carveout_indices).cpu().numpy())
            class EnsembleWrapper(nn.Module):
                def __init__(self, models_list, temps_list):
                    super().__init__()
                    self.models = nn.ModuleList(models_list)
                    self.temps = temps_list
                def forward(self, x):
                    probs = torch.zeros(x.size(0), N_CLASSES).to(x.device)
                    for i, m in enumerate(self.models):
                        probs += torch.softmax(m(x) / self.temps[i], dim=1)
                    return probs / len(self.models)
            
            model = EnsembleWrapper(ensemble_models, fold_temps)
            model.eval()
            print(f"\n[Info] Benchmark inferenza: Misurazione reale sull'Ensemble di {len(ensemble_models)} modelli.\n")
        else:
            model.eval()
            with torch.no_grad():
                for imgs, labels in final_test_loader:
                    imgs = imgs.to(device)
                    outputs = model(imgs)
                    y_true.extend(labels.numpy())
                    y_pred_raw.extend(outputs.argmax(dim=1).cpu().numpy())
                    y_pred.extend(calib.apply_reject_routing(outputs, best_t, best_tau, undiff_idx, carveout_indices).cpu().numpy())
        y_true = np.array(y_true)
        y_pred_raw = np.array(y_pred_raw)
        y_pred = np.array(y_pred)
        
        exp_manager.save_confusion_matrix(y_true, y_pred_raw, CLASS_NAMES, exp_dir, model_name + "_raw")
        report_raw = exp_manager.save_classification_report(y_true, y_pred_raw, CLASS_NAMES, exp_dir, filename="classification_report_raw.txt")

        exp_manager.save_confusion_matrix(y_true, y_pred, CLASS_NAMES, exp_dir, model_name + "_osr")
        report = exp_manager.save_classification_report(y_true, y_pred, CLASS_NAMES, exp_dir, filename="classification_report_osr.txt")
        
        inference_resources = trainer.benchmark_inference(model, final_test_loader, phase_name="inference")
        exp_manager.save_resource_usage(trainer.resource_tracker.records, exp_dir)
        
        ba_raw = balanced_accuracy_score(y_true, y_pred_raw)
        ba = balanced_accuracy_score(y_true, y_pred)
        with open(exp_dir / 'final_test_metrics.json', 'w') as f:
            json.dump({"test_balanced_accuracy_raw": ba_raw, "test_balanced_accuracy": ba, "best_t_calcolato": best_t, "best_tau_calcolato": best_tau}, f)
        print(f"\n{'='*40}")
        print(f"Balanced Accuracy (Argmax puro): {ba_raw*100:.2f}%")
        print(f"Balanced Accuracy (OSR routing): {ba*100:.2f}%")
        print(f"Miglioramento OSR: {(ba - ba_raw)*100:+.2f}%")
        fps = inference_resources.get('samples_per_second')
        peak_mb = inference_resources.get('gpu_peak_allocated_mb')
        if fps is not None:
            print(f"Inferenza: {fps:.2f} immagini/s")
        if peak_mb is not None:
            print(f"Picco GPU inferenza: {peak_mb:.1f} MB")
        print(f"Risorse salvate in: {exp_dir / 'logs' / 'resource_usage.json'}")
        print(f"\n=== REPORT ARGMAX (Senza OSR) ===\n{report_raw}")
        print(f"\n=== REPORT OSR (Con Reject) ===\n{report}")

    print(f"\n{'='*60}")
    total_elapsed = time.time() - total_start_time
    m, s = divmod(total_elapsed, 60)
    h, m = divmod(m, 60)
    print(f"Tempo totale richiesto: {int(h)}h {int(m)}m {int(s)}s")
    print(f"TRAINING COMPLETATO")
    print(f"Risultati salvati in: {exp_dir.parent}")
    print(f"{'='*60}")

    if IN_COLAB:
        print("\n[Info] Copia dei risultati su Google Drive in corso...")
        import shutil
        shutil.copytree(Path('/content/experiments'), BASE_DIR / 'experiments', dirs_exist_ok=True)
        print("[Info] Copia completata con successo.")


if not IN_COLAB:
    train_button = widgets.Button(
        description='Avvia Training',
        button_style='success',
        layout=widgets.Layout(width='200px', height='50px')
    )
    
    stop_button = widgets.Button(
        description='Interrompi',
        button_style='danger',
        disabled=True,
        layout=widgets.Layout(width='200px', height='50px')
    )

    training_output = widgets.Output(layout={'border': '1px solid gray'})
    
    def on_train_clicked(b):
        with training_output:
            training_output.clear_output(wait=True)
            try:
                train_button.disabled = True
                stop_button.disabled = False
                try:
                    run_training()
                except KeyboardInterrupt:
                    print("\n[!] Addestramento Globale Interrotto: run terminata, gli ultimi checkpoint sono salvi.")
            finally:
                train_button.disabled = False
                stop_button.disabled = True

    train_button.on_click(on_train_clicked)
    
    def on_stop_clicked(b):
        if 'trainer' in globals():
            trainer.stop_requested = True
            with training_output:
                print("\nRichiesta di interruzione ricevuta. Il training si fermerà...")
                
    stop_button.on_click(on_stop_clicked)
    
    display(widgets.VBox([
        widgets.HTML('<h3>Avvio Training</h3>'),
        widgets.HBox([train_button, stop_button]),
        training_output
    ]))
else:
    from IPython.display import display, HTML
    display(HTML('<p><b>Ambiente Colab rilevato:</b> i widget interattivi sono disabilitati. Esegui la cella successiva per avviare l\'addestramento in modalità bloccante nativa.</p>'))


In [ ]:
if IN_COLAB:
    try:
        run_training()
    except KeyboardInterrupt:
        print("\n[!] Addestramento Globale Interrotto: run terminata, gli ultimi checkpoint sono salvi.")


## 7 Visualizza Risultati

In [ ]:
# Elenca gli esperimenti salvati e permette di visualizzarli singolarmente tramite un menu a tendina.
from IPython.display import display, Image

experiments_dir = Path(output_base_dir.value) if 'output_base_dir' in globals() else Path('./experiments')

exp_dropdown = widgets.Dropdown(
    options=[],
    description='Esperimento:',
    layout=widgets.Layout(width='600px'),
    style={'description_width': 'initial'}
)

refresh_button = widgets.Button(
    description='Aggiorna Lista',
    button_style='info'
)

results_output = widgets.Output()

def get_experiment_list():
    current_dir = Path(output_base_dir.value) if 'output_base_dir' in globals() else Path('./experiments')
    if not current_dir.exists():
        return []
    
    exp_dirs = sorted(
        [d for d in current_dir.iterdir() if d.is_dir()],
        key=lambda x: x.stat().st_mtime,
        reverse=True
    )
    return [(d.name, d) for d in exp_dirs]

def update_dropdown(b=None):
    exps = get_experiment_list()
    if exps:
        exp_dropdown.options = exps
        exp_dropdown.value = exps[0][1]
    else:
        exp_dropdown.options = [("Nessun esperimento trovato", None)]
        exp_dropdown.value = None

def show_experiment_details(change=None):
    with results_output:
        clear_output()
        exp_dir = exp_dropdown.value
        
        if not exp_dir:
            return
            
        print(f"\n{'='*60}")
        print(f"ESPERIMENTO: {exp_dir.name}")
        print(f"{'='*60}")
        
        config_path = exp_dir / 'config.yaml'
        if config_path.exists():
            with open(config_path) as f:
                config = yaml.safe_load(f)
            model_name = config.get('model', {}).get('name', 'unknown')
            print(f"Modello utilizzato: {model_name}\n")
            print('--- CONFIGURAZIONE ESPERIMENTO ---')
            print(yaml.dump(config, default_flow_style=False, sort_keys=False).strip())
            print('----------------------------------\n')
            
            # 1. Balanced Accuracy (Honest Test Performance)
            test_metrics_path = exp_dir / 'final_test_metrics.json'
            if test_metrics_path.exists():
                with open(test_metrics_path) as f:
                    test_metrics = json.load(f)
                ba_raw = test_metrics.get("test_balanced_accuracy_raw")
                ba = test_metrics.get("test_balanced_accuracy", 0.0)
                if ba_raw is not None:
                    print(f"Test Balanced Accuracy (Argmax base): {ba_raw*100:.2f}%")
                print(f"Test Balanced Accuracy (OSR routing): {ba*100:.2f}%")
                if ba_raw is not None:
                    print(f"Delta OSR: {(ba - ba_raw)*100:+.2f}%\n")
                else:
                    print()

            else:
                history_files = list((exp_dir / 'logs').glob('*history*.json'))
                best_acc = 0
                for hf in history_files:
                    with open(hf) as f:
                        hist = json.load(f)
                    if 'val_bal_acc' in hist:
                        best_acc = max(best_acc, max(hist['val_bal_acc']))
                print(f"[Warning] Test metrics not found. Best Val Acc: {best_acc*100:.2f}%\n")
            
            # 2. Risorse hardware
            resource_path = exp_dir / 'logs' / 'resource_usage.json'
            if resource_path.exists():
                with open(resource_path) as f:
                    resources = json.load(f)
                inference = next((r for r in reversed(resources) if 'inference' in r.get('phase', '')), None)
                if inference:
                    fps = inference.get('samples_per_second')
                    peak = inference.get('gpu_peak_allocated_mb')
                    if fps is not None:
                        print(f"Velocità di inferenza: {fps:.2f} immagini/s")
                    if peak is not None:
                        print(f"Picco memoria GPU: {peak:.1f} MB")
            
            print(f"{'-'*60}\n")
            
            # 3. Report di classificazione
            report_path = exp_dir / 'logs' / 'classification_report.txt'
            if report_path.exists():
                print("CLASSIFICATION REPORT")
                with open(report_path, 'r') as f:
                    print(f.read())
                print(f"{'-'*60}\n")
            
            # 4. Immagini e Matrici di confusione
            plot_dir = exp_dir / 'plots'
            if plot_dir.exists():
                png_files = list(plot_dir.glob('*.png'))
                if png_files:
                    print("GRAFICI E MATRICI DI CONFUSIONE")
                    for img_path in png_files:
                        print(f"\n> {img_path.name}")
                        display(Image(filename=str(img_path)))
        else:
            print("Nessun file config.yaml trovato in questa cartella.")

refresh_button.on_click(update_dropdown)
exp_dropdown.observe(show_experiment_details, names='value')

# Inizializzazione
update_dropdown()

display(widgets.VBox([
    widgets.HTML('<h3>Risultati Esperimenti</h3>'),
    widgets.HBox([exp_dropdown, refresh_button]),
    results_output
]))


## 8 Analisi degli Errori con Grad-CAM

Questo blocco permette di ispezionare visivamente gli errori commessi dal modello sul Test Set. Tramite la **Grad-CAM** possiamo generare delle *heatmaps* (mappe di calore) che ci mostrano esattamente su quali parti dell'immagine la rete neurale si è concentrata prima di effettuare la sua predizione (errata).

Seleziona l'esperimento nel menu a tendina della cella precedente, e poi clicca il bottone qui sotto per avviare l'analisi.

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets
import torch
import numpy as np
import matplotlib.pyplot as plt
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision.transforms.functional import to_pil_image
import yaml
from pathlib import Path
from waste_classifier.trainer import ModelFactory
try:
    from torchcam.methods import GradCAM
    from torchcam.utils import overlay_mask
except ImportError:
    print("Installazione torchcam mancante.")

gradcam_output = widgets.Output(layout={'border': '1px solid gray'})

btn_run = widgets.Button(description='Avvia Analisi Grad-CAM', button_style='warning', layout=widgets.Layout(width='300px'))
btn_prev = widgets.Button(description='⏪ Precedenti 5', disabled=True)
btn_next = widgets.Button(description='Successive 5 ⏭️', disabled=True)
page_label = widgets.HTML(value='')
nav_box = widgets.HBox([btn_prev, page_label, btn_next])
nav_box.layout.display = 'none'

gradcam_state = {
    'errati': [], 'veri': [], 'predetti': [],
    'current_page': 0,
    'model': None,
    'device': None,
    'target_layers': [],
    'test_ds': None,
    'class_names': []
}

def render_page():
    with gradcam_output:
        clear_output(wait=True)
        
        errati = gradcam_state['errati']
        if not errati:
            return
            
        page = gradcam_state['current_page']
        start_idx = page * 5
        end_idx = min(start_idx + 5, len(errati))
        
        print(f"Mostrando errori da {start_idx+1} a {end_idx} su {len(errati)}...")
        
        model = gradcam_state['model']
        device = gradcam_state['device']
        target_layers = gradcam_state['target_layers']
        test_ds = gradcam_state['test_ds']
        class_names = gradcam_state['class_names']
        veri = gradcam_state['veri']
        predetti = gradcam_state['predetti']
        
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)
        
        for i in range(start_idx, end_idx):
            idx_reale = errati[i]
            vero_idx = veri[i]
            pred_idx = predetti[i]
            
            img_tensor, _ = test_ds[idx_reale]
            img_tensor = img_tensor.to(device)
            input_tensor = img_tensor.unsqueeze(0).requires_grad_(True)
            
            img_denorm = img_tensor * std + mean
            img_denorm = torch.clamp(img_denorm, 0, 1)
            img_pil = to_pil_image(img_denorm.cpu())
            
            fig, axes = plt.subplots(1, len(target_layers) + 1, figsize=(4 + 4*len(target_layers), 4))
            axes[0].imshow(img_pil)
            axes[0].set_title(f"Vero: {class_names[vero_idx]}\n Predetto: {class_names[pred_idx]}", color='red')
            axes[0].axis('off')
            
            for j, layer_name in enumerate(target_layers):
                cam_extractor = None
                try:
                    cam_extractor = GradCAM(model, target_layer=layer_name)
                    out = model(input_tensor)
                    activation_map = cam_extractor(int(pred_idx), out)
                    risultato = overlay_mask(img_pil, to_pil_image(activation_map[0].squeeze(0), mode='F'), alpha=0.5)
                    axes[j+1].imshow(risultato)
                    axes[j+1].set_title(f"Layer: {layer_name}")
                except Exception as e:
                    axes[j+1].text(0.5, 0.5, "N/A", ha='center', va='center')
                finally:
                    if cam_extractor is not None: cam_extractor.remove_hooks()
                    axes[j+1].axis('off')
                    
            plt.tight_layout()
            plt.show()
            
        # Aggiorna bottoni
        btn_prev.disabled = (page == 0)
        btn_next.disabled = (end_idx >= len(errati))
        page_label.value = f"&nbsp;&nbsp;<b>Pagina {page+1} / {(len(errati)-1)//5 + 1}</b>&nbsp;&nbsp;"

def analizza_errori_gradcam(b):
    nav_box.layout.display = 'none'
    with gradcam_output:
        clear_output(wait=True)
        exp_dir = exp_dropdown.value
        if not exp_dir: return print("Seleziona un esperimento dal menu a tendina.")
            
        config_path = exp_dir / 'config.yaml'
        if not config_path.exists(): return print(f"Errore: config.yaml non trovato in {exp_dir}")
            
        with open(config_path) as f:
            config = yaml.safe_load(f)
            
        model_name = config.get('model', {}).get('name')
        dataset_path = config.get('dataset', {}).get('path')
        if not model_name or not dataset_path: return print("Configurazione incompleta.")
            
        print(f"Avvio analisi Grad-CAM per il modello {model_name}... attendere.")
        
        try:
            full_dataset = ImageFolder(dataset_path, transform=None)
            targets = np.array(full_dataset.targets)
            class_names = full_dataset.classes
        except Exception as e:
            return print(f"Errore caricamento dataset: {e}")
            
        test_indices = []
        try:
            import csv
            splits_path = exp_dir / "dataset_splits.csv"
            if not splits_path.exists(): splits_path = exp_dir / "dataset_splits_ensemble.csv"
            if not splits_path.exists(): splits_path = exp_dir / "dataset_splits_final.csv"
            with open(splits_path, 'r', encoding='utf-8') as f:
                reader = csv.reader(f)
                next(reader)
                for i, row in enumerate(reader):
                    if 'test' in row[1].lower() or row[3] == 'True': test_indices.append(i)
        except Exception as e:
            return print("Errore caricamento indici di test dal CSV.")

        print(f"Test set selezionato: {len(test_indices)} immagini")
        
        from waste_classifier.trainer import AdaptiveAugmentationDataset
        from torch.utils.data import Subset
        input_size = 224 if 'efficientnet' not in model_name else (300 if 'b3' in model_name else (260 if 'b2' in model_name else 224))
        test_ds = AdaptiveAugmentationDataset(Subset(full_dataset, test_indices), {i: n.lower() for i, n in enumerate(class_names)}, resize=input_size+32, crop=input_size, is_train=False)
        test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = ModelFactory.create_model(model_name, num_classes=len(class_names), device=device, pretrained=False, dropout=0.0)
        
        model_path = exp_dir / "models" / "final_ft.pth"
        if not model_path.exists():
            model_path = exp_dir / "models" / "fold1_ft.pth"
        if not model_path.exists(): return print("Nessun modello addestrato trovato.")
            
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        
        print("Ricerca delle immagini classificate erroneamente...")
        veri, predetti, errati = [], [], []
        with torch.no_grad():
            for i, (inputs, labels) in enumerate(test_loader):
                outputs = model(inputs.to(device))
                _, preds = torch.max(outputs, 1)
                
                v = labels.numpy()
                p = preds.cpu().numpy()
                
                for j in range(len(v)):
                    idx_globale = i * 32 + j
                    if v[j] != p[j]:
                        errati.append(idx_globale)
                        veri.append(v[j])
                        predetti.append(p[j])

        if not errati: return print("Nessun errore riscontrato!")
        print(f"Trovati {len(errati)} errori.")

        if 'efficientnet' in model_name:
            target_layers = ['features.4', 'features.6', 'features.8']
        elif 'mobilenet' in model_name:
            target_layers = ['features.10', 'features.13']
        else:
            target_layers = [list(dict(model.named_modules()).keys())[-2]]

        gradcam_state.update({
            'errati': errati, 'veri': veri, 'predetti': predetti,
            'current_page': 0, 'model': model, 'device': device,
            'target_layers': target_layers, 'test_ds': test_ds, 'class_names': class_names
        })
        
        nav_box.layout.display = 'flex'
        nav_box.layout.justify_content = 'center'
        nav_box.layout.margin = '10px 0'
        
    render_page()

def on_prev(b):
    if gradcam_state['current_page'] > 0:
        gradcam_state['current_page'] -= 1
        render_page()

def on_next(b):
    if (gradcam_state['current_page'] + 1) * 5 < len(gradcam_state['errati']):
        gradcam_state['current_page'] += 1
        render_page()

btn_run.on_click(analizza_errori_gradcam)
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)

display(widgets.VBox([
    widgets.HTML('<h3>Analisi Errori (Grad-CAM) Interattiva</h3>'),
    widgets.HTML('<p>Usa questo strumento per esplorare gli errori del modello. Esegui la ricerca, poi naviga tra le pagine.</p>'),
    btn_run,
    nav_box,
    gradcam_output
]))


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import requests
from io import BytesIO
from PIL import Image
import torch
import torch.nn.functional as F
import yaml
import json

# Widget input URL
url_textarea = widgets.Textarea(
    value="https://img.magnific.com/foto-gratuito/adorabile-cane-basenji-marrone-e-bianco-che-sorride-e-che-da-il-cinque-isolato-su-bianco_346278-1657.jpg\nhttps://www.osservatoriodellaplastica.com/wp-content/uploads/2022/11/AdobeStock_323435465-scaled.jpeg",
    description='URL (uno per riga):',
    layout=widgets.Layout(width='800px', height='100px'),
    style={'description_width': 'initial'}
)

# Widget parametri (se non salvati, l'utente li mette a mano)
t_widget = widgets.FloatText(value=1.0, description='Temp (T):')
tau_widget = widgets.FloatText(value=0.5, description='Soglia (tau):')

test_url_button = widgets.Button(
    description='Carica Modello & Testa URL',
    button_style='info',
    layout=widgets.Layout(width='300px', height='40px')
)

url_output = widgets.Output(layout={'border': '1px solid gray'})

def on_test_url_clicked(b):
    with url_output:
        clear_output()
        
        # 1. Recupero cartella esperimento selezionata nella cella superiore
        if 'exp_dropdown' not in globals() or not exp_dropdown.value:
            print("ERRORE: Esegui la cella degli Esperimenti e seleziona un esperimento dal menu a tendina!")
            return
            
        exp_dir = exp_dropdown.value
        config_path = exp_dir / 'config.yaml'
        if not config_path.exists():
            print(f"File config.yaml non trovato in {exp_dir}")
            return
            
        with open(config_path) as f:
            config = yaml.safe_load(f)
            
        model_name = config.get('model', {}).get('name')
        if not model_name:
            print("Nome modello mancante nel config.")
            return
            
        # 2. Cerco best_tau e best_t nel JSON (se li abbiamo salvati in addestramenti futuri)
        best_t, best_tau = t_widget.value, tau_widget.value
        metrics_path = exp_dir / 'final_test_metrics.json'
        loaded_from_json = False
        if metrics_path.exists():
            with open(metrics_path, 'r') as f:
                metrics = json.load(f)
                if 'best_tau_calcolato' in metrics and 'best_t_calcolato' in metrics:
                    best_tau = metrics['best_tau_calcolato']
                    best_t = metrics['best_t_calcolato']
                    loaded_from_json = True
                    # Aggiorno i widget per mostrare all'utente i valori trovati
                    t_widget.value = best_t
                    tau_widget.value = best_tau

        # 3. Caricamento Modello
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Caricamento modello {model_name} dall'esperimento '{exp_dir.name}'...")
        try:
            from waste_classifier.trainer import ModelFactory
            my_classes = CLASS_NAMES if 'CLASS_NAMES' in globals() else ['battery', 'clothing', 'glass', 'metal', 'organic', 'papery', 'plastic', 'undifferentiated']
            model = ModelFactory.create_model(model_name, len(my_classes), device, pretrained=False)
            model_path = exp_dir / "models" / "final_ft.pth"
            if not model_path.exists():
                model_path = exp_dir / "models" / "fold1_ft.pth"
            model.load_state_dict(torch.load(model_path, map_location=device))
            model.eval()
        except Exception as e:
            print(f"Errore caricamento pesi: {e}")
            return
            
        urls = [u.strip() for u in url_textarea.value.split('\n') if u.strip()]
        if not urls:
            print("Nessun URL inserito.")
            return
            
        print(f"\n--- TEST OOD ---")
        if loaded_from_json:
            print(f"Parametri recuperati automaticamente dall'esperimento: T={best_t:.4f}, tau={best_tau:.4f}\n")
        else:
            print(f"ATTENZIONE: T e tau NON trovati nel file di log di questo vecchio esperimento.")
            print(f"Uso i parametri manuali inseriti qui sopra: T={best_t:.4f}, tau={best_tau:.4f}\n")
        
        print(f"{'psi_a':>7}  {'ARGMAX':<15} {'ROUTED (reject)':<18}  {'TOP-3 PROBABILITA'}")
        print("-" * 110)
        
        import torchvision.transforms as transforms
        from waste_classifier.calibration import apply_reject_routing
        
        size = INPUT_SIZES.get(model_name, 224) if 'INPUT_SIZES' in globals() else 224
        preprocess = transforms.Compose([
            transforms.Resize(size + 32),
            transforms.CenterCrop(size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        
        # Recupera indici se non ci sono (fallback)
        my_undiff = undiff_idx if 'undiff_idx' in globals() else 7
        my_carveout = carveout_indices if 'carveout_indices' in globals() else [6, 2, 3] # plastic, glass, metal
        
        for url in urls:
            try:
                headers = {"User-Agent": "Mozilla/5.0"}
                r = requests.get(url, headers=headers, timeout=10)
                r.raise_for_status()
                img = Image.open(BytesIO(r.content)).convert('RGB')
                
                x = preprocess(img).unsqueeze(0).to(device)
                with torch.no_grad():
                    logits = model(x)
                    probs = F.softmax(logits / best_t, dim=1)[0].cpu()
                
                argmax_idx = int(probs.argmax())
                routed_idx = int(apply_reject_routing(logits, best_t, best_tau, my_undiff, my_carveout)[0].item())
                psi_a = float(probs.max())
                
                top3 = torch.topk(probs, 3)
                tops_str = ", ".join(f"{my_classes[i]}={p:.2f}" for p, i in zip(top3.values.tolist(), top3.indices.tolist()))
                
                display(img.resize((150, 150)))
                print(f"{psi_a:>7.3f}  {my_classes[argmax_idx]:<15} {my_classes[routed_idx]:<18}  {tops_str}")
                print(f"URL: {url[:80]}...\n")
                
            except Exception as e:
                print(f"Errore caricamento URL: {url}\n{e}\n")

test_url_button.on_click(on_test_url_clicked)

# Layout per T e tau
params_box = widgets.HBox([t_widget, tau_widget])

display(widgets.VBox([
    widgets.HTML('<h3>🔎 Test Manuale URL e Probabilità OOD (Caricamento Esperimenti)</h3>'),
    widgets.HTML('<p>Incolla gli URL delle immagini. Il codice userà il <b>modello selezionato nel menu a tendina dell\'esperimento precedente</b>.</p>'),
    params_box,
    url_textarea,
    test_url_button,
    url_output
]))
